# 01: The Math of Attention (QK-Norm)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/01_the_math_of_attention.ipynb)

[← Previous: 00 Setup & Tokenization](00_setup_and_tokenization.ipynb) | [Next: 02 Causal Masking →](02_causal_masking.ipynb)


**Estimated Time: 15 minutes**

In this notebook, we implement the **core mechanism** of Gemma 3's decoder: **Query-Key Attention**. The key architectural shift in Gemma 3 is replacing the soft-capping (tanh at 50.0) of Gemma 2 with **QK-Norm** — L2-normalizing both Q and K before computing dot products.

---
## Learning Objectives
1. Understand the Query (Q), Key (K), and Value (V) analogy.
2. Implement attention with **QK-Norm** (Gemma 3's core innovation).
3. Understand why scaling by $1/\sqrt{d_k}$ is still needed after QK-Norm.
4. Visualize how Softmax creates an attention map.
5. Understand why final logits are capped at 30.0.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Set seed for reproducibility
_ = torch.manual_seed(42)

# Gemma 3 architecture parameters
head_dim = 96  # hidden_size=768 / num_heads=8
seq_len = 8
batch_size = 1
logit_cap = 30.0  # Gemma 3 final logit capping

---
## 1. The Q, K, V Analogy

Imagine you are in a library:
- **Query (Q)**: The topic you are searching for (e.g., "How do transformers work?").
- **Key (K)**: The labels on the spines of the books (e.g., "Deep Learning", "Gardening", "Attention Mechanisms").
- **Value (V)**: The actual content inside the books.

Attention matches your **Query** against all **Keys**, computes a similarity score, and uses that score to decide how much of each **Value** you should read.

In Gemma 2, attention scores were constrained using a scalar tangent function. In **Gemma 3**, we use **QK-Norm**: normalize Q and K vectors to unit length before the dot product, making scores purely cosine similarities.

In [2]:
# Create dummy Q, K, V tensors
# Shape: (batch, seq_len, head_dim)
Q = torch.randn(batch_size, seq_len, head_dim)
K = torch.randn(batch_size, seq_len, head_dim)
V = torch.randn(batch_size, seq_len, head_dim)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

# Verify they have the same head dimension
assert Q.shape[-1] == K.shape[-1] == V.shape[-1] == head_dim
print("✅ Shapes are consistent!")

---
## 🌐 Step-by-Step Mechanics: QK-Norm (Gemma 3 Innovation)

In **Gemma 2**, attention scores were scaled and capped using `tanh(scores / 50.0)`. In **Gemma 3**, Google introduces **QK-Norm** to stabilize attention values across deep layers and massive contexts. 

QK-Norm (Query-Key Normalization) is the idea of adding a normalization step before QK multiplication happens. By normalizing $Q$ and $K$ first, you guarantee that the resulting attention scores stay stable, which prevents the Softmax function from saturating and killing the training gradients.

When architects decide to implement QK-Norm, they have to choose how to normalize those vectors. They could use LayerNorm, L2 Normalization, or RMSNorm.

Crucially, Gemma 3 uses **RMSNorm (Root Mean Square Normalization)**, not L2 normalization.

### 🧮 Mathematical Principle:

1. L2 Normalization (The Unit Sphere)

L2 Normalization forces the entire vector to have a physical length (magnitude) of exactly 1.0.

$$L2(x) = \frac{x}{||x||_2}$$

If you apply this to the Queries and Keys, the maximum possible dot product between them becomes 1.0.

2. RMSNorm (Root Mean Square)

RMSNorm does not force the length of the vector to 1.0. Instead, it forces the variance of the numbers inside the vector to 1.0.

Before computing dot products, we apply RMSNorm to $Q$ and $K$ along their final dimension (the head dimension, $d_k$). 
First, we calculate the Root Mean Square of the vector:

$$RMS(v) = \sqrt{\frac{1}{d_k} \sum_{i=1}^{d_k} v_i^2 + \epsilon}$$

Then we scale the vectors by the inverse of this RMS:

$$\text{RMSNorm}(x) = \frac{v}{\sqrt{\frac{1}{d_k} \sum v_i^2 + \epsilon}} \times \gamma$$

Because it divides by the mean squared, the physical length of the resulting vector scales with the square root of the dimensions ($\sqrt{d}$). It also includes a trainable weight parameter ($\gamma$) that allows the network to learn exactly how loud that specific head should be.

> **Note:** Softmax thrives on variance. It needs strong, distinct "spikes" to decisively pick one word over another. If you use pure L2 Normalization, the maximum dot-product score is 1.0. When you feed scores bounded between -1 and 1 into Softmax, the resulting probabilities are incredibly "soft" and flat. The model gets confused and tries to pay equal attention to every word in the context window. Because RMSNorm scales the vector by $\sqrt{d_{head}}$ (which is $\sqrt{96} \approx 9.79$ in Gemma 3), the dot products are much larger. This gives Softmax the mathematically sharp spikes it needs to focus 99% of its attention on exactly the right word.

### 🔍 Architectural Impact:
- **Variance Stabilization**: RMSNorm forces the individual elements of the $Q$ and $K$ vectors to have a variance of exactly 1. 
- **Perfect Scaling Synergy**: Because the variance is 1, the overall magnitude (length) of the vectors becomes $\sqrt{d_k}$. When we compute the dot product and subsequently scale it by $1/\sqrt{d_k}$, the final attention logits are perfectly standardized back to a variance of ~1. This prevents the Softmax distribution from flattening out (uniform noise) or saturating (vanishing gradients)!

Hence:
RMSNorm does not force the vector length to 1. Instead, it forces the variance of the vector's elements to 1. Mathematically, this means the overall length (magnitude) of the vector actually becomes $\sqrt{d_k}$. When you take the dot product of two vectors with magnitude $\sqrt{d_k}$, the resulting score scales naturally. Then, when you apply the standard transformer division by $\sqrt{d_k}$, it perfectly scales the variance of the logits back to exactly 1.0.

In [15]:
import torch
import torch.nn.functional as F
import math


def rms_norm(tensor, eps=1e-6):
    """Simple RMSNorm without learnable weights for QK-Norm."""
    return tensor * torch.rsqrt(tensor.pow(2).mean(-1, keepdim=True) + eps)


# QK-Norm: RMS normalize Q and K
Q_norm = rms_norm(Q)
K_norm = rms_norm(K)

# Calculate expected magnitude (sqrt(d_k))
expected_magnitude = math.sqrt(head_dim)

print(f"Expected vector magnitude: ~{expected_magnitude:.4f}")
print(f"Actual Q_norm magnitude:   {Q_norm.norm(dim=-1)[0, 0]:.4f}")
print(f"Actual K_norm magnitude:   {K_norm.norm(dim=-1)[0, 0]:.4f}")

print(f"\nQ_norm mean squared: {Q_norm.pow(2).mean(dim=-1)[0, 0]:.4f}")
print(f"Q_norm variance: {Q_norm.var(dim=-1)[0, 0]:.4f}")

# Verify RMS normalization worked (mean squared value should be exactly 1)
assert torch.allclose(
    Q_norm.pow(2).mean(dim=-1), torch.ones_like(Q_norm.pow(2).mean(dim=-1)), atol=1e-4
)
print(
    "\n✅ QK-Norm applied: Vectors have a mean squared value of 1, preparing them perfectly for scaled dot-products!"
)

# Verify RMS normalization worked (variance should be roughly 1)
assert torch.allclose(
    Q_norm.var(dim=-1, unbiased=False), torch.ones_like(Q_norm.var(dim=-1)), atol=0.05
)
print(
    "✅ QK-Norm applied: Vectors have a variance of ~1, preparing them perfectly for scaled dot-products!"
)

---
## 📊 Attention Score Matrix Multiplication & Scaling

With normalized $Q$ and $K$ tensors, we now compute the attention scores. This represents the strength of connection between every position in the query sequence and every position in the key sequence.

### 📐 Dimensional Analysis & Tensor Shapes:
- $Q_{norm}$ shape: `(batch_size, seq_len, head_dim)` $\rightarrow$ `(1, 8, 96)`
- $K_{norm}$ shape: `(batch_size, seq_len, head_dim)` $\rightarrow$ `(1, 8, 96)`
- **Transposition**: To multiply Query and Key along the head dimension, we transpose the last two dimensions of $K_{norm}$ to shape `(1, 96, 8)`.
- **Dot Product**:

  $$\text{Scores} = \frac{Q_{norm} K_{norm}^T}{\sqrt{d_k}} = \frac{(1, 8, 96) \times (1, 96, 8)}{\sqrt{96}} = (1, 8, 8)$$

  *(This results in an $8 \times 8$ matrix representing attention scores between all 8 tokens).*

  Here is the final attention formula:

  $$Attention = \text{Softmax}\left(\frac{\text{Norm}(Q) \cdot \text{Norm}(K)^T}{\sqrt{d_k}}\right) \cdot V$$

### Gemma 3's Specific Implementation

To make things even more stable, Gemma 3 uses a highly specific flavor of RMSNorm throughout its architecture (both in the main transformer blocks and the QK-Norm). Instead of standard weights, Gemma 3 uses zero-centered weights. The scaling parameter ($\gamma$) is initialized at pure 0.0, and the math looks like this:

$$\text{Output} = \text{RMSNorm}(x) \times (1 + \text{weight})$$

This ensures that at the exact moment the model is initialized (Step Zero), the QK-Norm does absolutely nothing to alter the shape of the vectors — acting as a perfect identity pass-through. As training progresses, the model slowly learns to adjust the weights away from zero, organically shaping the attention landscape without causing the gradient shocks that plague massive 128K-context models.

### ⚡ Scaling Factor ($1/\sqrt{d_k}$):
Even with QK-norm, we scale by $1/\sqrt{d_k}$ (where $d_k = 96$ is the head dimension). Scaling by $\sqrt{96} \approx 9.798$ standardizes variance, maintaining optimal softmax sensitivity.

In [4]:
# Compute attention scores with QK-Norm
scores = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / math.sqrt(head_dim)

print(f"Scores shape: {scores.shape}")
print(f"Score range: [{scores.min():.4f}, {scores.max():.4f}]")
print(f"\nSample scores (first query vs all keys):\n{scores[0, 0]}")

# With QK-Norm, scores are naturally bounded (unlike raw dot products)
import math

print(f"\nScore std without normalization: ~{scores.std().item():.4f}")
print("With QK-Norm, scores are much more stable!")

This standardizes the variance, ensuring the subsequent Softmax operation remains sensitive and doesn't collapse into a hard one-hot distribution.

---
## 4. Softmax and Attention Weights

In [5]:
# Apply softmax to get attention weights
attention_weights = F.softmax(scores, dim=-1)

print("Attention Weights (Sum of row 1):", attention_weights[0, 0].sum().item())
print("\nThese weights sum to 1.0 -- they are a probability distribution.")

# Visualized as a heatmap
plt.figure(figsize=(8, 3))
plt.imshow(attention_weights[0].detach().numpy(), cmap="viridis")
plt.colorbar()
plt.title("Attention Map: Where each position focuses")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.tight_layout()
plt.show()

The Softmax function (`F.softmax`) acts as a strict accountant. It mathematically forces all of those wild scores into a neat probability distribution.

Every score is squashed into a decimal between 0.0 and 1.0 (acting as a percentage).

Because of `dim=-1` (which tells it to calculate across the horizontal row of Keys), the sum of the entire row is forced to equal exactly 1.0 (or 100%).

This demonstrates that attention is a zero-sum game. If the current Query wants to pay 90% of its attention to the word "Apple," it only has 10% left to divide among the rest of the sentence.

---
## 5. The Final Output

In [6]:
# Weighted sum of Value vectors
output = torch.matmul(attention_weights, V)
print(f"Output shape: {output.shape}")

# The output is a context-aware representation of each position
print(f"\nFirst token's context vector (first 5 dims):\n{output[0, 0, :5]}")

At its absolute core, the dot product $Q \cdot K^T$ represents the Relevance Score between two words.

Imagine the Attention mechanism is a filing cabinet system:
* The Query ($Q$): The current word you are processing asks a question. For example, if the current word is "chased," its Query vector might mathematically represent: "I am an action verb. I am looking for a furry noun that does the chasing."
* The Key ($K$): Every word in the sentence (including the current word) holds up a nametag describing what it is. The word "Dog" holds up a Key vector that says: "I am a furry noun."
  
The Dot Product is the act of comparing the Query to the Key. 

It mathematically measures how well the "question" matches the "nametag".
- High Positive Score (e.g., +8.5): A perfect match! The model should pay a lot of attention here.
- Near Zero (e.g., 0.1): Completely unrelated. Ignore it.
- Large Negative Score (e.g., -5.0): Actively opposite or contradictory. Do not look here.

If the Dot Product ($Q \cdot K^T$) is the "interview process," and Softmax is the "accountant" dividing up the budget, then multiplying by $V$ (the Value) is the final payload.

To finish our analogy from earlier:
- The Query ($Q$): The question you are asking. (e.g., "I need the object of this verb.")
- The Key ($K$): The label on the outside of the filing cabinet folder. (e.g., "I am a noun.")
- The Value ($V$): The actual documents inside the folder. 

The Key and the Value usually start from the exact same place (the word's embedding), but they go through different filters (projections). The Key is filtered to be a highly searchable nametag. The Value is filtered to contain the deep, underlying meaning of the word.

---
## 🛡️ Gemma 3 Final Logit Soft-Capping

While **QK-Norm** handles normalization inside the attention block, Gemma 3 still applies a **final logit soft-capping** operation at the very end of the network (after the linear LM head projections). 

### 🧮 Soft-Capping Formula:

$$\text{Logits}_{capped} = C \times \tanh\left(\frac{\text{Logits}}{C}\right)$$

Where $C = 30.0$ is the logit capping threshold.

### 🧠 Why do we apply Logit Capping?
- **Prevents Softmax Overconfidence**: When raw logits grow extremely large (e.g. $>50$), the softmax distribution becomes too "peaky", assigning a probability of $1.0$ to one token and $0.0$ to others. This completely stops gradient signals from propagating during training.
- **Taming Extrapolations**: Soft-capping via $\tanh$ restricts the range of logits smoothly to $[-30.0, 30.0]$. At value $30$, $\tanh(30/30) = \tanh(1) \approx 0.761$, capping the output at $30 \times 0.761 \approx 22.8$. This prevents logits from exploding while preserving rank relationships.

In [7]:
def final_logit_cap(logits, cap=logit_cap):
    """Gemma 3 final logit soft-capping."""
    return cap * torch.tanh(logits / cap)


# Demonstrate the effect
raw_logits = torch.randn(1, 100) * 50  # Unbounded logits
capped_logits = final_logit_cap(raw_logits)

print(f"Raw logits:     min={raw_logits.min():.1f}, max={raw_logits.max():.1f}")
print(f"Capped logits:  min={capped_logits.min():.1f}, max={capped_logits.max():.1f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(raw_logits[0].detach().numpy(), bins=50, color="steelblue")
plt.title("Raw Logits (unbounded)")
plt.xlabel("Logit Value")

plt.subplot(1, 2, 2)
plt.hist(capped_logits[0].detach().numpy(), bins=50, color="coral")
plt.title("Capped Logits (tanh, cap=30)")
plt.xlabel("Logit Value")
plt.tight_layout()
plt.show()

---
## Exercise

Write a single function `qk_attention(Q, K, V)` that performs QK-Norm + scaled dot-product attention. Then test it matches the step-by-step computation above.

In [10]:
def qk_attention(Q, K, V):
    """Attention with QK-Norm (Gemma 3 style)."""
    d_k = Q.shape[-1]

    # 1. QK-Norm: L2 normalize both Q and K
    # TODO: Implement the Gemma 3 normalization trick here!
    Q_norm = rms_norm(Q)
    K_norm = rms_norm(K)

    if Q_norm is Ellipsis:
        raise NotImplementedError("Implement QK-Norm before continuing!")

    # 2. Compute dot product scores, scaled by 1/sqrt(dk)
    scores = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / math.sqrt(d_k)

    # 3. Softmax over the last dimension
    attention_weights = F.softmax(scores, dim=-1)

    # 4. Weighted sum of values
    output = torch.matmul(attention_weights, V)

    return output


# Test your function
test_output = qk_attention(Q, K, V)
assert torch.allclose(test_output, output, atol=1e-6)
print("✅ Success! Your QK-Norm attention matches the expected output.")

<details>
<summary><b>Click to see solution</b></summary>

```python
def rms_norm(tensor, eps=1e-6):
    """Simple RMSNorm without learnable weights for QK-Norm."""
    return tensor * torch.rsqrt(tensor.pow(2).mean(-1, keepdim=True) + eps)

def qk_attention(Q, K, V):
    d_k = Q.shape[-1]
    
    # Apply RMSNorm instead of L2 Norm!
    Q_norm = rms_norm(Q)
    K_norm = rms_norm(K)
    
    scores = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / math.sqrt(d_k)
    attention_weights = F.softmax(scores, dim=-1)
    
    return torch.matmul(attention_weights, V)
```
</details>

---
## Summary

Gemma 3 replaced Gemma 2's soft-capping with QK-Norm — a cleaner way to stabilize attention. The 30.0 final logit capping remains, preventing extreme confidence at the output layer.

[← Previous: 00 Setup & Tokenization](00_setup_and_tokenization.ipynb) | [Next: 02 Causal Masking →](02_causal_masking.ipynb)